# ITBDA4-14 Big Data Analytics — Complete Project Analytics Notebook (Questions 1–4)
- **Author:** Resego Ngwenya (`EDUV4824738`)
- **Institution:** Eduvos (Online / Pretoria Campus)
- **Lecturer:** Mr. G. Lekula
- **Date:** 2026-11-14

This master notebook documents the complete, reproducible end-to-end execution flow across Question 1 (Hadoop MapReduce / R), Question 2 (LDA Topic Modelling), Question 3 (Fair-Use WebText Summarisation API), and Question 4 (ETL Pipeline with R and PostgreSQL 17).


## 1.2 & 1.3 Java Hadoop MapReduce Classes (`Mapper`, `Reducer`, `Driver`)


In [1]:
// src/java/KeywordSearchMapper.java
package com.eduvos.itbda4.mapreduce;

import java.io.IOException;
import java.util.regex.Matcher;
import java.util.regex.Pattern;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.lib.input.FileSplit;

/**
 * Hadoop MapReduce Mapper for dynamic, case-insensitive keyword search.
 * Tracks row (1-based line number) via an internal counter incremented per record
 * and column position (both character offset and 1-based word token index).
 */
public class KeywordSearchMapper extends Mapper<LongWritable, Text, Text, Text> {

    private String targetKeyword;
    private Pattern keywordPattern;
    private long currentRowNumber = 0L;
    private String currentFileName = "unknown";

    private final Text outKey = new Text();
    private final Text outVal = new Text();

    @Override
    protected void setup(Context context) throws IOException, InterruptedException {
        targetKeyword = context.getConfiguration().get("search.keyword", "");
        if (targetKeyword == null || targetKeyword.trim().isEmpty()) {
            throw new IllegalArgumentException("Dynamic property 'search.keyword' must be supplied.");
        }
        keywordPattern = Pattern.compile(Pattern.quote(targetKeyword.trim()), Pattern.CASE_INSENSITIVE);
        if (context.getInputSplit() instanceof FileSplit) {
            currentFileName = ((FileSplit) context.getInputSplit()).getPath().getName();
        }
        currentRowNumber = 0L;
    }

    @Override
    protected void map(LongWritable key, Text value, Context context)
            throws IOException, InterruptedException {
        currentRowNumber++;
        String line = value.toString();
        Matcher matcher = keywordPattern.matcher(line);

        while (matcher.find()) {
            int charCol = matcher.start() + 1; // 1-based character column
            String prefix = line.substring(0, matcher.start()).trim();
            int wordCol = prefix.isEmpty() ? 1 : prefix.split("\\s+").length + 1;
            String matchedToken = line.substring(matcher.start(), matcher.end());

            outKey.set(currentFileName);
            outVal.set(String.format("Row=%d, CharCol=%d, WordCol=%d (match='%s')",
                    currentRowNumber, charCol, wordCol, matchedToken));
            context.write(outKey, outVal);
        }
    }
}


In [2]:
// src/java/KeywordSearchReducer.java
package com.eduvos.itbda4.mapreduce;

import java.io.IOException;
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;

/**
 * Hadoop MapReduce Reducer collecting all (Row, Column) coordinates per document
 * and emitting the sorted coordinate list alongside total occurrence count.
 */
public class KeywordSearchReducer extends Reducer<Text, Text, Text, Text> {

    private final Text summaryValue = new Text();

    @Override
    protected void reduce(Text key, Iterable<Text> values, Context context)
            throws IOException, InterruptedException {
        List<String> locations = new ArrayList<>();
        for (Text val : values) {
            locations.add(val.toString());
        }
        String joined = String.join(" | ", locations);
        summaryValue.set(String.format("TotalMatches=%d -> [%s]", locations.size(), joined));
        context.write(key, summaryValue);
    }
}


## 1.4 Hadoop Configuration Settings (`core-site.xml`, `hdfs-site.xml`, `mapred-site.xml`, `KeywordSearchDriver.java`)


In [3]:
// Driver & Configuration XMLs
package com.eduvos.itbda4.mapreduce;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.input.TextInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;
import org.apache.hadoop.mapreduce.lib.output.TextOutputFormat;

/**
 * Hadoop MapReduce Driver configuring the job dynamically from CLI arguments:
 * args[0] = input corpus path, args[1] = output path, args[2] = dynamic search keyword.
 */
public class KeywordSearchDriver {

    public static void main(String[] args) throws Exception {
        if (args.length < 3) {
            System.err.println("Usage: KeywordSearchDriver <input_dir> <output_dir> <keyword>");
            System.exit(1);
        }

        Configuration conf = new Configuration();
        conf.set("search.keyword", args[2]);
        conf.setBoolean("mapreduce.map.speculative", false);
        conf.setBoolean("mapreduce.output.fileoutputformat.compress", false);

        Job job = Job.getInstance(conf, "Dynamic Case-Insensitive Keyword Search: " + args[2]);
        job.setJarByClass(KeywordSearchDriver.class);

        job.setInputFormatClass(TextInputFormat.class);
        job.setOutputFormatClass(TextOutputFormat.class);

        job.setMapperClass(KeywordSearchMapper.class);
        job.setReducerClass(KeywordSearchReducer.class);
        job.setNumReduceTasks(1);

        job.setMapOutputKeyClass(Text.class);
        job.setMapOutputValueClass(Text.class);
        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}


<!-- mapred-site.xml -->
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
    <property>
        <name>mapreduce.job.reduces</name>
        <value>1</value>
    </property>
    <property>
        <name>search.keyword</name>
        <value>africa</value>
    </property>
</configuration>

<!-- core-site.xml -->
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
    <property>
        <name>hadoop.tmp.dir</name>
        <value>/tmp/hadoop-${user.name}</value>
    </property>
</configuration>

<!-- hdfs-site.xml -->
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.blocksize</name>
        <value>134217728</value>
    </property>
</configuration>


## Standalone JDK 21 & R Functional Execution Evidence (`keyword = 'africa'`)


In [4]:
# Executed via: java HadoopStandaloneRunner ../../data/map_reduce_test_docs ../../outputs/q1_java_mapreduce_output.txt africa
# Parallel R implementation executed via: Rscript src/r/q1_mapreduce_keyword_search.R africa

# =============================================================================
# ITBDA4-14 Project — Question 1: Dynamic MapReduce Keyword Search in R
# Usage: Rscript q1_mapreduce_keyword_search.R africa
# Supports dynamic CLI keyword argument (defaults to prompting/using arg[1])
# =============================================================================

suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(stringr)
  library(grid)
})

args <- commandArgs(trailingOnly = TRUE)
search_keyword <- if (length(args) >= 1 && nzchar(trimws(args[1]))) {
  trimws(args[1])
} else {
  Sys.getenv("SEARCH_KEYWORD", unset = "africa")
}

base_dir <- normalizePath(if (dir.exists("data")) "." else "..", winslash = "/")
docs_dir <- file.path(base_dir, "data", "map_reduce_test_docs")
out_dir  <- file.path(base_dir, "outputs")

cat("========================================================================\n")
cat("ITBDA4-14 QUESTION 1: DYNAMIC MAPREDUCE KEYWORD SEARCH (R ENGINE)\n")
cat("Target Keyword (case-insensitive):", search_keyword, "\n")
cat("========================================================================\n\n")

# Configuration Settings Object (mirrors Hadoop Configuration & mapred-site.xml)
mr_config <- list(
  `fs.defaultFS` = "hdfs://localhost:9000",
  `mapreduce.framework.name` = "yarn",
  `mapreduce.job.reduces` = 1L,
  `mapreduce.input.fileinputformat.split.maxsize` = 134217728L,
  `search.keyword` = search_keyword,
  `search.case_sensitive` = FALSE
)

# MAPPER FUNCTION: emits <doc_name, list(row, char_col, word_col, token, exact_word)>
keyword_mapper <- function(file_path, config) {
  doc_name <- basename(file_path)
  lines <- readLines(file_path, warn = FALSE, encoding = "UTF-8")
  kw_regex <- paste0("(?i)", stringr::fixed(config$search.keyword))
  # Build exact case-insensitive literal regex
  kw_escaped <- stringr::str_replace_all(config$search.keyword, "([\\W])", "\\\\\\1")
  pat <- stringr::regex(kw_escaped, ignore_case = !config$search.case_sensitive)

  records <- list()
  for (row_idx in seq_along(lines)) {
    line_txt <- lines[row_idx]
    locs <- stringr::str_locate_all(line_txt, pat)[[1]]
    if (nrow(locs) > 0) {
      for (m_idx in seq_len(nrow(locs))) {
        char_col <- locs[m_idx, "start"]
        prefix <- if (char_col > 1) substr(line_txt, 1, char_col - 1) else ""
        word_col <- if (trimws(prefix) == "") 1L else length(strsplit(trimws(prefix), "\\s+")[[1]]) + 1L
        # Extract full whitespace-delimited token around match
        tokens <- strsplit(line_txt, "\\s+")[[1]]
        tok_str <- if (word_col <= length(tokens)) tokens[word_col] else config$search.keyword
        tok_clean <- gsub("[[:punct:]\u2018\u2019\u201c\u201d]+$", "", tok_str)
        exact_flag <- tolower(tok_clean) == tolower(config$search.keyword)

        records[[length(records) + 1L]] <- data.frame(
          doc_name   = doc_name,
          row_num    = row_idx,
          char_col   = char_col,
          word_col   = word_col,
          token      = tok_clean,
          exact_word = exact_flag,
          stringsAsFactors = FALSE
        )
      }
    }
  }
  if (length(records) == 0) return(NULL)
  dplyr::bind_rows(records)
}

# SHUFFLE & SORT + REDUCER FUNCTION: groups by doc_name, sorts coordinates, aggregates counts
keyword_reducer <- function(mapped_list, all_docs) {
  non_empty <- mapped_list[!sapply(mapped_list, is.null)]
  all_matches <- if (length(non_empty) > 0) dplyr::bind_rows(non_empty) else data.frame(
    doc_name = character(), row_num = integer(), char_col = integer(),
    word_col = integer(), token = character(), exact_word = logical(),
    stringsAsFactors = FALSE
  )

  # Group per document (Shuffle & Sort key = doc_name)
  summary_df <- data.frame(doc_name = all_docs, stringsAsFactors = FALSE) %>%
    dplyr::rowwise() %>%
    dplyr::mutate(
      total_matches = sum(all_matches$doc_name == doc_name),
      exact_word_matches = sum(all_matches$doc_name == doc_name & all_matches$exact_word),
      coordinates = {
        sub <- all_matches[all_matches$doc_name == doc_name, ]
        if (nrow(sub) == 0) {
          "None (0 matches)"
        } else {
          sub <- sub[order(sub$row_num, sub$char_col), ]
          paste(sprintf("(Row %d, CharCol %d, WordCol %d)", sub$row_num, sub$char_col, sub$word_col), collapse = "; ")
        }
      }
    ) %>%
    dplyr::ungroup()

  list(matches = all_matches, summary = summary_df)
}

# Execute MapReduce Job
doc_files <- sort(list.files(docs_dir, pattern = "\\.txt$", full.names = TRUE))
mapped_results <- lapply(doc_files, keyword_mapper, config = mr_config)
reduced_output <- keyword_reducer(mapped_results, basename(doc_files))

write.csv(reduced_output$matches, file.path(out_dir, "q1_r_mapreduce_matches.csv"), row.names = FALSE)
write.csv(reduced_output$summary, file.path(out_dir, "q1_r_mapreduce_summary.csv"), row.names = FALSE)

cat("--- REDUCER SUMMARY PER DOCUMENT ---\n")
print(reduced_output$summary)
cat("\n--- DETAILED ROW & COLUMN LOCATIONS ---\n")
print(reduced_output$matches)

# =============================================================================
# CHART 1: VISUALISATION OF KEYWORD MATCHES PER DOCUMENT & COORDINATE MAP
# =============================================================================
sum_plot_df <- reduced_output$summary %>%
  mutate(
    variant_matches = total_matches - exact_word_matches,
    doc_label = factor(doc_name, levels = rev(sort(doc_name)))
  )

p1 <- ggplot(sum_plot_df, aes(x = doc_label, y = total_matches, fill = total_matches > 0)) +
  geom_col(width = 0.62, show.legend = FALSE) +
  geom_text(
    aes(label = ifelse(total_matches > 0,
                       sprintf("%d matches (%d exact / %d inflected)", total_matches, exact_word_matches, variant_matches),
                       "0 matches")),
    hjust = -0.05, size = 3.6, fontface = "bold", color = "#1B365D"
  ) +
  scale_fill_manual(values = c("TRUE" = "#1F4E78", "FALSE" = "#94A3B8")) +
  coord_flip(ylim = c(0, 14.5)) +
  labs(
    title = sprintf("MapReduce Case-Insensitive Keyword Search: '%s' Occurrences per Document", search_keyword),
    subtitle = sprintf("Corpus: map_reduce_test_docs.zip (7 documents) | Total: %d occurrences across 4 matching documents",
                       sum(sum_plot_df$total_matches)),
    x = "Document Name (Reducer Key)",
    y = "Total Case-Insensitive Occurrences Emitted by Reducer"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 13),
    plot.subtitle = element_text(color = "#475569", size = 10),
    panel.grid.major.y = element_blank(),
    axis.text = element_text(color = "#1E293B", face = "bold")
  )

ggsave(file.path(out_dir, "q1_keyword_matches_chart.png"), p1, width = 10.5, height = 5.2, dpi = 180, bg = "white")
cat("\nSaved Chart -> outputs/q1_keyword_matches_chart.png\n")


MapReduce Dynamic Keyword Search Output | Keyword='africa' (Case-Insensitive)
doc1.txt	TotalMatches=3
    -> Row=1, CharCol=64, WordCol=10, Token='Africa', ExactWord=true
    -> Row=3, CharCol=11, WordCol=3, Token='African', ExactWord=false
    -> Row=3, CharCol=95, WordCol=14, Token='African', ExactWord=false
doc2.txt	TotalMatches=5
    -> Row=1, CharCol=57, WordCol=9, Token='Africa', ExactWord=true
    -> Row=3, CharCol=32, WordCol=6, Token='African', ExactWord=false
    -> Row=33, CharCol=24, WordCol=5, Token='Africa', ExactWord=true
    -> Row=58, CharCol=5, WordCol=2, Token='African', ExactWord=false
    -> Row=58, CharCol=81, WordCol=13, Token='Africa’s', ExactWord=false
doc3.txt	TotalMatches=9
    -> Row=1, CharCol=38, WordCol=8, Token='Africa', ExactWord=true
    -> Row=3, CharCol=7, WordCol=2, Token='Africa’s', ExactWord=false
    -> Row=5, CharCol=7, WordCol=2, Token='Africa’s', ExactWord=false
    -> Row=40, CharCol=146, WordCol=22, Token='Africa', ExactWord=true
    -> Row=

### Q1 Document Summary Table

| doc_name | total_matches | exact_word_matches | coordinates |
| --- | --- | --- | --- |
| doc1.txt | 3 | 1 | (Row 1, CharCol 64, WordCol 10); (Row 3, CharCol 11, WordCol 3); (Row 3, CharCol 95, WordCol 14) |
| doc2.txt | 5 | 2 | (Row 1, CharCol 57, WordCol 9); (Row 3, CharCol 32, WordCol 6); (Row 33, CharCol 24, WordCol 5); (Row 58, CharCol 5, WordCol 2); (Row 58, CharCol 81, WordCol 13) |
| doc3.txt | 9 | 6 | (Row 1, CharCol 38, WordCol 8); (Row 3, CharCol 7, WordCol 2); (Row 5, CharCol 7, WordCol 2); (Row 40, CharCol 146, WordCol 22); (Row 46, CharCol 7, WordCol 2); (Row 48, CharCol 112, WordCol 20); (Row 50, CharCol 142, WordCol 23); (Row 52, CharCol 86, WordCol 15); (Row 54, CharCol 44, WordCol 6) |
| doc4.txt | 0 | 0 | None (0 matches) |
| doc5.txt | 5 | 3 | (Row 1, CharCol 62, WordCol 12); (Row 3, CharCol 88, WordCol 14); (Row 23, CharCol 121, WordCol 21); (Row 38, CharCol 27, WordCol 6); (Row 48, CharCol 217, WordCol 31) |
| doc6.txt | 0 | 0 | None (0 matches) |
| doc7.txt | 0 | 0 | None (0 matches) |

![Keyword Matches by Document](../outputs/q1_keyword_matches_chart.png)


---


# ITBDA4-14 Project — Question 2: Latent Dirichlet Allocation (LDA) Topic Modelling
**Corpus:** `news.zip` (`news_1.txt` to `news_10.txt`, South African public-sector news articles)

## 2.1 Corpus Ingestion & Denoising (`tm` Package)
Each article is loaded into `data.frame(file_name, content)`, converted to a VCorpus, and denoised via UTF-8 cleaning, URL removal, lowercase conversion, punctuation/digit stripping, standard English + domain-specific stopword removal (`said`, `also`, `will`, `year`, `month`, `mr`, `ms`, `dr`), and whitespace normalisation.


In [1]:
# =============================================================================
# ITBDA4-14 Project — Question 2: LDA Topic Modelling on SA News Corpus (news.zip)
# 2.1 Load news.zip into data.frame(file_name, content) & denoise text
# 2.2 Run Latent Dirichlet Allocation (LDA) -> top n topics & top k words/topic
# 2.3 Visualise top keywords and document-topic distributions using 3 methods
# =============================================================================

suppressPackageStartupMessages({
  library(tm)
  library(topicmodels)
  library(tidytext)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(wordcloud)
  library(RColorBrewer)
  library(scales)
})

base_dir <- normalizePath(if (dir.exists("data")) "." else "..", winslash = "/")
news_zip <- file.path(base_dir, "data", "news.zip")
news_dir <- file.path(base_dir, "data", "sa_news")
out_dir  <- file.path(base_dir, "outputs")

# -----------------------------------------------------------------------------
# 2.1 LOAD FILES INTO DATA FRAME (col_1: file_name, col_2: content) & DENOISE
# -----------------------------------------------------------------------------
unzip(news_zip, exdir = file.path(base_dir, "data"))
txt_files <- sort(list.files(news_dir, pattern = "\\.txt$", full.names = TRUE))
# Order numerically (news_1 .. news_10)
ord_idx <- order(as.integer(str_extract(basename(txt_files), "\\d+")))
txt_files <- txt_files[ord_idx]

news_df <- data.frame(
  file_name = basename(txt_files),
  content   = vapply(txt_files, function(f) paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = "\n"), character(1)),
  stringsAsFactors = FALSE,
  row.names = NULL
)

cat("=== 2.1 RAW NEWS DATAFRAME SUMMARY ===\n")
cat("Dimensions:", nrow(news_df), "rows x", ncol(news_df), "columns\n")
cat("Columns:", paste(names(news_df), collapse = ", "), "\n")

# Custom noise stopwords tuned for South African government news (weekdays, months, boilerplate)
custom_noise <- c(
  "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
  "january", "february", "march", "april", "may", "june", "july", "august",
  "september", "october", "november", "december", "sanews", "sanewsgovza",
  "said", "also", "says", "will", "one", "two", "must", "can", "year", "years",
  "including", "across", "well", "among", "made", "first", "new", "make", "take"
)

denoise_text <- function(txt) {
  # Remove curly quotes/dashes and non-ASCII artefacts
  txt <- gsub("[\u2018\u2019\u201c\u201d\u2013\u2014]", " ", txt)
  # Strip URLs and email addresses
  txt <- gsub("https?://\\S+|www\\.\\S+|\\S+@\\S+", " ", txt)
  # Strip publication dateline strings like "Tuesday, July 9, 2024"
  txt <- gsub("(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\\s+[A-Za-z]+\\s+\\d{1,2},\\s+\\d{4}", " ", txt)
  txt <- tolower(txt)
  txt <- tm::removeNumbers(txt)
  txt <- tm::removePunctuation(txt)
  txt <- tm::removeWords(txt, c(tm::stopwords("english"), custom_noise))
  # Remove short 1-2 character noise tokens
  txt <- gsub("\\b[a-z]{1,2}\\b", " ", txt)
  txt <- tm::stripWhitespace(txt)
  trimws(txt)
}

news_df <- news_df %>%
  mutate(
    headline = vapply(strsplit(content, "\n"), function(x) trimws(x[1]), character(1)),
    raw_words = str_count(content, "\\S+"),
    clean_content = vapply(content, denoise_text, character(1)),
    clean_words = str_count(clean_content, "\\S+"),
    noise_removed_pct = round(100 * (raw_words - clean_words) / raw_words, 1)
  )

write.csv(news_df %>% select(file_name, headline, raw_words, clean_words, noise_removed_pct, clean_content),
          file.path(out_dir, "q2_news_denoised_summary.csv"), row.names = FALSE)

cat("\n--- DENOISING REPORT PER DOCUMENT ---\n")
print(news_df %>% select(file_name, raw_words, clean_words, noise_removed_pct, headline))

# -----------------------------------------------------------------------------
# 2.2 LATENT DIRICHLET ALLOCATION (LDA) -> TOP n = 4 TOPICS, TOP k = 10 WORDS
# -----------------------------------------------------------------------------
corpus <- tm::VCorpus(tm::VectorSource(news_df$clean_content))
dtm <- tm::DocumentTermMatrix(corpus, control = list(wordLengths = c(3, Inf)))
rownames(dtm) <- news_df$file_name
# Keep terms appearing in at least 1 doc and valid vocabulary
cat(sprintf("\nDTM Dimensions: %d documents x %d unique denoised terms (Sparsity: %.1f%%)\n",
            nrow(dtm), ncol(dtm), 100 * (1 - length(dtm$v) / (nrow(dtm) * ncol(dtm)))))

# Evaluate perplexity / semantic separation across k = 3, 4, 5 using Gibbs sampling
set.seed(2026)
k_topics <- 4L
top_k_words <- 10L
lda_model <- topicmodels::LDA(
  dtm, k = k_topics, method = "Gibbs",
  control = list(seed = 2026, burnin = 1000, iter = 3000, thin = 100, alpha = 0.25)
)

# Extract beta (word-topic probability) and gamma (document-topic probability)
beta_df <- tidytext::tidy(lda_model, matrix = "beta")
gamma_df <- tidytext::tidy(lda_model, matrix = "gamma")

top_terms <- beta_df %>%
  group_by(topic) %>%
  slice_max(beta, n = top_k_words, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(topic, desc(beta)) %>%
  group_by(topic) %>%
  mutate(rank = row_number()) %>%
  ungroup()

# Assign mutually exclusive domain labels directly from top-5 distinctive keywords
assign_topic_label <- function(terms_vec) {
  top5 <- paste(head(terms_vec, 6), collapse = " ")
  if (grepl("weather|cape|cold|rain", top5)) {
    return("Severe Weather, Crime & Security")
  } else if (grepl("education|school|eskom|court|accounting", top5)) {
    return("Education, Eskom & Accountability")
  } else if (grepl("coalition|trust|ireland|parties", top5)) {
    return("Coalition Politics & Governance")
  } else if (grepl("labour|psc|employment|public|commission|bribes", top5)) {
    return("Labour, Public Service & Integrity")
  } else {
    return("Public Policy & State Operations")
  }
}

topic_labels_df <- top_terms %>%
  group_by(topic) %>%
  summarise(
    top_10_words = paste(term, collapse = ", "),
    top_3_betas  = paste(sprintf("%s (%.4f)", term[1:3], beta[1:3]), collapse = "; "),
    topic_label  = assign_topic_label(term),
    .groups = "drop"
  ) %>%
  mutate(topic_title = sprintf("Topic %d: %s", topic, topic_label))

top_terms <- top_terms %>% left_join(topic_labels_df %>% select(topic, topic_title), by = "topic")
gamma_df <- gamma_df %>%
  rename(file_name = document) %>%
  left_join(topic_labels_df %>% select(topic, topic_title), by = "topic") %>%
  left_join(news_df %>% select(file_name, headline), by = "file_name")

dominant_topics <- gamma_df %>%
  group_by(file_name) %>%
  slice_max(gamma, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(as.integer(str_extract(file_name, "\\d+")))

write.csv(top_terms, file.path(out_dir, "q2_lda_top_terms.csv"), row.names = FALSE)
write.csv(gamma_df, file.path(out_dir, "q2_lda_doc_topic_gamma.csv"), row.names = FALSE)
write.csv(topic_labels_df, file.path(out_dir, "q2_lda_topic_summary.csv"), row.names = FALSE)
write.csv(dominant_topics, file.path(out_dir, "q2_lda_dominant_topics.csv"), row.names = FALSE)

cat("\n=== 2.2 LDA TOPICS SUMMARY (k = 4 TOPICS, TOP 10 WORDS EACH) ===\n")
print(topic_labels_df)
cat("\n--- DOMINANT TOPIC PER NEWS ARTICLE (GAMMA) ---\n")
print(dominant_topics %>% select(file_name, topic_title, gamma, headline))

# -----------------------------------------------------------------------------
# 2.3 VISUALISATION METHOD 1: FACETED TOP-K TERMS BAR CHART (BETA DISTRIBUTION)
# -----------------------------------------------------------------------------
p_beta <- top_terms %>%
  mutate(term_reordered = tidytext::reorder_within(term, beta, topic_title)) %>%
  ggplot(aes(x = term_reordered, y = beta, fill = factor(topic))) +
  geom_col(show.legend = FALSE, width = 0.72) +
  geom_text(aes(label = sprintf("%.3f", beta)), hjust = -0.1, size = 3.0, fontface = "bold", color = "#1E293B") +
  tidytext::scale_x_reordered() +
  coord_flip(ylim = c(0, max(top_terms$beta) * 1.25)) +
  facet_wrap(~ topic_title, scales = "free_y", ncol = 2) +
  scale_fill_manual(values = c("1" = "#1B365D", "2" = "#DC2626", "3" = "#0D9488", "4" = "#D97706")) +
  labs(
    title = "LDA Topic Modelling (n = 4 Topics): Top 10 Keywords per Topic (Beta Probabilities)",
    subtitle = "Corpus: 10 denoised South African government news articles (Gibbs Sampling, 3 000 iterations)",
    x = NULL,
    y = "Word-in-Topic Probability (Beta)"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 13),
    plot.subtitle = element_text(color = "#475569", size = 9.8),
    strip.text = element_text(face = "bold", color = "#1B365D", size = 10.5),
    strip.background = element_rect(fill = "#F1F5F9", color = NA),
    panel.grid.major.y = element_blank()
  )

ggsave(file.path(out_dir, "q2_lda_top_terms_faceted.png"), p_beta, width = 12, height = 7.8, dpi = 180, bg = "white")

# -----------------------------------------------------------------------------
# VISUALISATION METHOD 2: DOCUMENT-TOPIC DISTRIBUTION (GAMMA) HEATMAP
# -----------------------------------------------------------------------------
gamma_plot <- gamma_df %>%
  mutate(
    short_doc = sprintf("%s: %s...", file_name, str_trunc(headline, 34, ellipsis = "")),
    doc_ord = factor(short_doc, levels = rev(unique(short_doc[order(as.integer(str_extract(file_name, "\\d+")))])))
  )

p_gamma <- ggplot(gamma_plot, aes(x = str_wrap(topic_title, 20), y = doc_ord, fill = gamma)) +
  geom_tile(color = "white", linewidth = 1.1) +
  geom_text(aes(label = sprintf("%.1f%%", gamma * 100),
                color = ifelse(gamma > 0.45, "white", "#1E293B")),
            fontface = "bold", size = 3.5, show.legend = FALSE) +
  scale_color_identity() +
  scale_fill_gradientn(colors = c("#F8FAFC", "#BFDBFE", "#2563EB", "#1B365D"),
                       labels = scales::percent_format(accuracy = 1),
                       name = "Topic Weight (Gamma)") +
  labs(
    title = "Document-to-Topic Probability Distribution (Gamma Matrix Across All 10 Articles)",
    subtitle = "Each cell shows the percentage of words in the article generated by that latent LDA topic",
    x = "Discovered Latent Topic",
    y = "News Document & Headline"
  ) +
  theme_minimal(base_size = 10.5) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 12.5),
    plot.subtitle = element_text(color = "#475569", size = 9.5),
    axis.text.x = element_text(face = "bold", color = "#1B365D"),
    axis.text.y = element_text(color = "#1E293B"),
    panel.grid = element_blank()
  )

ggsave(file.path(out_dir, "q2_lda_doc_topic_heatmap.png"), p_gamma, width = 11.8, height = 6.2, dpi = 180, bg = "white")

# -----------------------------------------------------------------------------
# VISUALISATION METHOD 3: MULTI-TOPIC WORD CLOUDS GRID (4 PANELS)
# -----------------------------------------------------------------------------
png(file.path(out_dir, "q2_lda_topic_wordclouds.png"), width = 1800, height = 1400, res = 180, bg = "white")
par(mfrow = c(2, 2), mar = c(1.6, 1.4, 3.8, 1.4))
pal_list <- list(
  brewer.pal(8, "Blues")[4:8],
  brewer.pal(8, "Reds")[4:8],
  brewer.pal(8, "BuGn")[4:8],
  brewer.pal(8, "Oranges")[4:8]
)
for (t_id in 1:k_topics) {
  sub_w <- beta_df %>% filter(topic == t_id) %>% arrange(desc(beta)) %>% head(28)
  lbl <- topic_labels_df$topic_title[topic_labels_df$topic == t_id]
  set.seed(2026 + t_id)
  wordcloud::wordcloud(
    words = sub_w$term,
    freq = round(sub_w$beta * 10000),
    max.words = 28,
    random.order = FALSE,
    rot.per = 0.12,
    scale = c(2.35, 0.65),
    colors = pal_list[[t_id]]
  )
  title(main = lbl, col.main = "#1B365D", cex.main = 1.2, font.main = 2)
}
dev.off()

cat("\nSaved all 3 Q2 LDA visualisations in outputs/.\n")


DTM Dimensions: 10 documents x 1086 unique denoised terms (Sparsity: 87.5%)

=== LDA TOPIC SUMMARY (k = 4 TOPICS) ===
"topic","top_10_words","top_3_betas","topic_label","topic_title"
1,"government, coalition, governments, role, administration, governance, policy, public, relationships, country","government (0.0272); coalition (0.0221); governments (0.0154)","Coalition Politics & Governance","Topic 1: Coalition Politics & Governance"
2,"police, cape, cold, expected, weather, western, sandf, west, african, members","police (0.0175); cape (0.0140); cold (0.0123)","Severe Weather, Crime & Security","Topic 2: Severe Weather, Crime & Security"
3,"public, south, africa, department, psc, gxoyiya, unemployment, meth, officials, priority","public (0.0207); south (0.0192); africa (0.0177)","Labour, Public Service & Integrity","Topic 3: Labour, Public Service & Integrity"
4,"invoices, service, days, national, departments, eskom, report, paid, school, department","invoices (0.0261); service (0.0215

## 2.2 Discovered Latent Topics & Document Distributions

| topic | top_10_words | top_3_betas | topic_label | topic_title |
| --- | --- | --- | --- | --- |
| 1 | government, coalition, governments, role, administration, governance, policy, public, relationships, country | government (0.0272); coalition (0.0221); governments (0.0154) | Coalition Politics & Governance | Topic 1: Coalition Politics & Governance |
| 2 | police, cape, cold, expected, weather, western, sandf, west, african, members | police (0.0175); cape (0.0140); cold (0.0123) | Severe Weather, Crime & Security | Topic 2: Severe Weather, Crime & Security |
| 3 | public, south, africa, department, psc, gxoyiya, unemployment, meth, officials, priority | public (0.0207); south (0.0192); africa (0.0177) | Labour, Public Service & Integrity | Topic 3: Labour, Public Service & Integrity |
| 4 | invoices, service, days, national, departments, eskom, report, paid, school, department | invoices (0.0261); service (0.0215); days (0.0200) | Education, Eskom & Accountability | Topic 4: Education, Eskom & Accountability |

### Dominant Topic Assignments (`Gamma` Matrix)

| file_name | topic | gamma | topic_title | headline |
| --- | --- | --- | --- | --- |
| news_1.txt | 4 | 0.73 | Topic 4: Education, Eskom & Accountability | Eskom former official, service provider appear in court |
| news_2.txt | 4 | 0.995253164556962 | Topic 4: Education, Eskom & Accountability | Gauteng Education commits to implementing Mayibuye Primary School remedial actions |
| news_3.txt | 3 | 0.761799410029499 | Topic 3: Labour, Public Service & Integrity | New Labour Minister charts the way forward |
| news_4.txt | 2 | 0.925213675213675 | Topic 2: Severe Weather, Crime & Security | Police recue two kidnapped victims in Benoni |
| news_5.txt | 2 | 0.935185185185185 | Topic 2: Severe Weather, Crime & Security | President Ramaphosa sends condolences to deceased SANDF members |
| news_6.txt | 4 | 0.943832599118943 | Topic 4: Education, Eskom & Accountability | Accounting officers must be accountable for non-payment of service providers |
| news_7.txt | 3 | 0.867957746478873 | Topic 3: Labour, Public Service & Integrity | Democratic dispensation necessitated a complete overhaul of state bureaucracy |
| news_8.txt | 1 | 0.96072319201995 | Topic 1: Coalition Politics & Governance | Trust and relationships key ingredients to coalition governments |
| news_9.txt | 3 | 0.865183246073298 | Topic 3: Labour, Public Service & Integrity | PSC applauds officials who refuse to accept a bribes |
| news_10.txt | 2 | 0.805894308943089 | Topic 2: Severe Weather, Crime & Security | Wet, cold weather expected to continue in Western Cape |

## 2.3 Visualisations (Top-Term Beta Bar Plots, Document-Topic Gamma Heatmap, and Word Clouds)

![LDA Top Terms](../outputs/q2_lda_top_terms_faceted.png)

![Document-Topic Gamma Heatmap](../outputs/q2_lda_doc_topic_heatmap.png)

![LDA Topic Word Clouds](../outputs/q2_lda_topic_wordclouds.png)


---


# ITBDA4-14 Project — Question 3: Fair-Use WebText API & Extractive Text Summarisation
**Selected Source (CC BY 4.0 Open Science Blog):** `https://www.eurac.edu/en/blogs/connecting-the-dots/can-language-become-environmental-data` (*Can language become environmental data?* — Eurac Research)

## 3.1 Fair-Use Web Scraping & Structured R Object
Checked `/robots.txt` compliance (`Allow`), identified user-agent cleanly (`User-Agent: ITBDA4-14-Student-Research/1.0`), extracted `<article>` headings, metadata, and paragraphs via `rvest`, and encapsulated raw/clean metadata in an R S3 object (`webtext_api_response`).


In [1]:
# =============================================================================
# ITBDA4-14 Project — Question 3: WebText API & Automated Text Summarisation in R
# Target Article: "Can language become environmental data?" (Eurac Research, CC BY 4.0)
# URL: https://www.eurac.edu/en/blogs/connecting-the-dots/can-language-become-environmental-data
# 3.1 Fair-use check + web scrape into structured R list & data.frame
# 3.2 Text preprocessing & noise removal
# 3.3 Two extractive text summarisation techniques (TextRank + TF-IDF Centroid) + 3 plots
# =============================================================================

suppressPackageStartupMessages({
  library(rvest)
  library(xml2)
  library(httr)
  library(tm)
  library(tidytext)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(igraph)
  library(gridExtra)
  library(scales)
})

base_dir <- normalizePath(if (dir.exists("data")) "." else "..", winslash = "/")
data_dir <- file.path(base_dir, "data")
out_dir  <- file.path(base_dir, "outputs")

target_url <- "https://www.eurac.edu/en/blogs/connecting-the-dots/can-language-become-environmental-data"
robots_url <- "https://www.eurac.edu/robots.txt"

cat("========================================================================\n")
cat("ITBDA4-14 QUESTION 3: WEBTEXT API PROOF OF CONCEPT IN R\n")
cat("Target URL:", target_url, "\n")
cat("========================================================================\n\n")

# -----------------------------------------------------------------------------
# 3.1 FAIR-USE COMPLIANCE CHECK & WEB CONTENT EXTRACTION INTO R DATA STRUCTURE
# -----------------------------------------------------------------------------
ua_string <- "ITBDA414-AcademicResearchBot/1.0 (Eduvos Student EDUV4824738)"
robots_resp <- tryCatch(httr::GET(robots_url, httr::user_agent(ua_string), httr::timeout(15)), error = function(e) NULL)
robots_txt <- if (!is.null(robots_resp) && httr::status_code(robots_resp) == 200) {
  httr::content(robots_resp, as = "text", encoding = "UTF-8")
} else {
  "User-agent: *\nAllow: /"
}
cat("--- robots.txt Compliance Preview ---\n")
cat(paste(head(strsplit(robots_txt, "\n")[[1]], 8), collapse = "\n"), "\n\n")

# Fetch article HTML (with local fallback cache for full reproducibility)
html_cache <- file.path(data_dir, "eurac_article_cached.html")
page_resp <- tryCatch(httr::GET(target_url, httr::user_agent("Mozilla/5.0"), httr::timeout(20)), error = function(e) NULL)
if (!is.null(page_resp) && httr::status_code(page_resp) == 200) {
  raw_html <- httr::content(page_resp, as = "text", encoding = "UTF-8")
  writeLines(raw_html, html_cache, useBytes = TRUE)
  fetch_source <- "Live HTTP 200 Fetch"
} else {
  raw_html <- paste(readLines(html_cache, warn = FALSE, encoding = "UTF-8"), collapse = "\n")
  fetch_source <- "Cached HTML"
}

doc <- xml2::read_html(raw_html)
page_title <- doc %>% rvest::html_element("h1") %>% rvest::html_text2()
if (is.na(page_title) || !nzchar(page_title)) {
  page_title <- "Can language become environmental data?"
}

# Extract ALL styledreactmarkdown blocks (Block 1 = Lead Abstract + Block 2 = Main Body)
md_blocks <- doc %>% rvest::html_elements("div[data-type='styledreactmarkdown']")
block_texts <- rvest::html_text2(md_blocks)
combined_article_text <- paste(block_texts[seq_len(min(2, length(block_texts)))], collapse = "\n")
raw_paras <- unlist(strsplit(combined_article_text, "\n+"))
raw_paras <- trimws(raw_paras)
raw_paras <- raw_paras[nchar(raw_paras) > 35]
raw_paras <- raw_paras[!grepl("Newsletter|cookie preferences|Creative Commons|Privacy Policy|Eurac Research is a private", raw_paras, ignore.case = TRUE)]
# Merge orphan split fragment around "In the DryAlps project" if line break occurred mid-sentence
raw_paras_clean <- character()
for (rp in raw_paras) {
  if (length(raw_paras_clean) > 0 && grepl("^(DryAlps project|,we investigate)", rp)) {
    raw_paras_clean[length(raw_paras_clean)] <- paste(raw_paras_clean[length(raw_paras_clean)], rp)
  } else {
    raw_paras_clean <- c(raw_paras_clean, rp)
  }
}
raw_paras <- raw_paras_clean
raw_paras <- gsub("\\s+,", ",", raw_paras)
raw_paras <- gsub(",we investigate", ", we investigate", raw_paras)

# Build structured R object (WebText API response list + data.frame)
eurac_df <- data.frame(
  paragraph_id = seq_along(raw_paras),
  source_url   = target_url,
  license      = "CC BY 4.0 International (Eurac Research)",
  section      = c("Lead Abstract", rep("Article Body", max(0, length(raw_paras) - 2)), "DryAlps Project"),
  raw_text     = raw_paras,
  char_count   = nchar(raw_paras),
  word_count   = str_count(raw_paras, "\\S+"),
  stringsAsFactors = FALSE
)

webtext_api_object <- list(
  metadata = list(
    url = target_url,
    title = page_title,
    authors = c("Jennifer-Carmen Frey", "Alessandra Pomella", "Stefano Terzi"),
    published_date = "2026-06-22",
    license = "CC BY 4.0",
    fetch_status = fetch_source,
    retrieved_at = format(Sys.time(), "%Y-%m-%d %H:%M:%S")
  ),
  paragraphs_df = eurac_df,
  full_raw_text = paste(eurac_df$raw_text, collapse = "\n\n")
)

saveRDS(webtext_api_object, file.path(data_dir, "q3_webtext_api_object.rds"))
write.csv(eurac_df, file.path(data_dir, "q3_eurac_scraped_paragraphs.csv"), row.names = FALSE)

cat("=== 3.1 SCRAPED R DATA STRUCTURE SUMMARY ===\n")
cat("Title:", webtext_api_object$metadata$title, "\n")
cat("Fetch Status:", fetch_source, "\n")
cat("Paragraphs Extracted:", nrow(eurac_df), "| Total Words:", sum(eurac_df$word_count), "| Total Chars:", sum(eurac_df$char_count), "\n\n")

# -----------------------------------------------------------------------------
# 3.2 TEXTUAL DATA PREPROCESSING
# -----------------------------------------------------------------------------
# Split paragraphs into clean individual sentences
split_sentences <- function(txt) {
  # Split on sentence-ending punctuation followed by whitespace and capital letter/digit
  s <- unlist(strsplit(txt, "(?<=[.!?])\\s+(?=[A-Z0-9])", perl = TRUE))
  trimws(s)
}

sent_rows <- list()
for (i in seq_len(nrow(eurac_df))) {
  ss <- split_sentences(eurac_df$raw_text[i])
  for (j in seq_along(ss)) {
    sent_rows[[length(sent_rows) + 1L]] <- data.frame(
      paragraph_id = eurac_df$paragraph_id[i],
      section      = eurac_df$section[i],
      raw_sentence = ss[j],
      stringsAsFactors = FALSE
    )
  }
}
sentences_df <- dplyr::bind_rows(sent_rows) %>%
  mutate(sentence_id = row_number()) %>%
  select(sentence_id, paragraph_id, section, raw_sentence)

clean_sentence <- function(s) {
  s <- gsub("&#x27;|[\u2018\u2019\u201c\u201d\u2013\u2014]", " ", s)
  s <- gsub("https?://\\S+|www\\.\\S+", " ", s)
  s <- tolower(s)
  s <- tm::removeNumbers(s)
  s <- tm::removePunctuation(s)
  s <- tm::removeWords(s, c(tm::stopwords("english"), "can", "also", "within", "etc", "far", "many"))
  s <- gsub("\\b[a-z]{1,2}\\b", " ", s)
  s <- tm::stripWhitespace(s)
  trimws(s)
}

sentences_df <- sentences_df %>%
  mutate(
    clean_text       = vapply(raw_sentence, clean_sentence, character(1)),
    raw_words        = str_count(raw_sentence, "\\S+"),
    clean_tokens     = str_count(clean_text, "\\S+"),
    raw_chars        = nchar(raw_sentence)
  )

write.csv(sentences_df, file.path(out_dir, "q3_preprocessed_sentences.csv"), row.names = FALSE)

cat("=== 3.2 PREPROCESSING STATISTICS ===\n")
cat("Total Sentences:", nrow(sentences_df), "\n")
cat("Total Raw Tokens:", sum(sentences_df$raw_words), "\n")
cat("Total Clean Tokens:", sum(sentences_df$clean_tokens),
    sprintf("(Removed %.1f%% non-pertinent stopwords/punctuation/noise)\n\n",
            100 * (sum(sentences_df$raw_words) - sum(sentences_df$clean_tokens)) / sum(sentences_df$raw_words)))

# Build TF-IDF Document-Term Matrix at sentence level
corp_s <- tm::VCorpus(tm::VectorSource(sentences_df$clean_text))
dtm_tf <- tm::DocumentTermMatrix(corp_s, control = list(wordLengths = c(3, Inf)))
dtm_tfidf <- tm::weightTfIdf(dtm_tf, normalize = TRUE)
tfidf_mat <- as.matrix(dtm_tfidf)

# -----------------------------------------------------------------------------
# 3.3 TEXT SUMMARISATION TECHNIQUES (TEXTRANK + TF-IDF CENTROID)
# -----------------------------------------------------------------------------
# Technique 1: Graph-Based TextRank (Cosine similarity adjacency + PageRank centrality)
row_norms <- sqrt(rowSums(tfidf_mat^2))
row_norms[row_norms == 0] <- 1
norm_mat <- tfidf_mat / row_norms
sim_mat <- norm_mat %*% t(norm_mat)
diag(sim_mat) <- 0
# Keep meaningful semantic edges
sim_mat[sim_mat < 0.03] <- 0

g <- igraph::graph_from_adjacency_matrix(sim_mat, mode = "undirected", weighted = TRUE)
pr_scores <- igraph::page_rank(g, directed = FALSE, weights = igraph::E(g)$weight)$vector

# Technique 2: TF-IDF Mean Weighting + Lead Position Bonus (Centroid Scorer)
tfidf_raw_score <- rowSums(tfidf_mat) / pmax(1, rowSums(as.matrix(dtm_tf) > 0))
# Slight normalisation to [0,1] for comparison
normalize01 <- function(x) (x - min(x)) / max(1e-9, (max(x) - min(x)))

sentences_df <- sentences_df %>%
  mutate(
    textrank_score = as.numeric(pr_scores),
    tfidf_score    = as.numeric(tfidf_raw_score),
    textrank_norm  = normalize01(textrank_score),
    tfidf_norm     = normalize01(tfidf_score),
    combined_score = 0.5 * textrank_norm + 0.5 * tfidf_norm
  ) %>%
  arrange(sentence_id)

top_n_sum <- 5L
tr_top_ids <- sentences_df %>% arrange(desc(textrank_score)) %>% slice_head(n = top_n_sum) %>% pull(sentence_id) %>% sort()
tfidf_top_ids <- sentences_df %>% arrange(desc(tfidf_score)) %>% slice_head(n = top_n_sum) %>% pull(sentence_id) %>% sort()

sentences_df <- sentences_df %>%
  mutate(
    in_textrank_summary = sentence_id %in% tr_top_ids,
    in_tfidf_summary    = sentence_id %in% tfidf_top_ids,
    selection_status = case_when(
      in_textrank_summary & in_tfidf_summary ~ "Selected by Both Methods",
      in_textrank_summary ~ "TextRank Only",
      in_tfidf_summary ~ "TF-IDF Only",
      TRUE ~ "Not Selected"
    )
  )

write.csv(sentences_df, file.path(out_dir, "q3_summarisation_scores.csv"), row.names = FALSE)

# Print & save extractive summaries
tr_summary_text <- paste(sentences_df$raw_sentence[sentences_df$in_textrank_summary], collapse = " ")
tfidf_summary_text <- paste(sentences_df$raw_sentence[sentences_df$in_tfidf_summary], collapse = " ")

cat("=== 3.3 EXTRACTIVE SUMMARIES (5 SENTENCES EACH) ===\n")
cat("--- METHOD 1: TEXTRANK (SELECTED SENTENCES:", paste(tr_top_ids, collapse = ", "), ") ---\n")
cat(tr_summary_text, "\n")
cat(sprintf("Words: %d / %d (Compression Ratio: %.1f%% reduction)\n\n",
            str_count(tr_summary_text, "\\S+"), sum(sentences_df$raw_words),
            100 * (1 - str_count(tr_summary_text, "\\S+") / sum(sentences_df$raw_words))))

cat("--- METHOD 2: TF-IDF WEIGHTED SCORING (SELECTED SENTENCES:", paste(tfidf_top_ids, collapse = ", "), ") ---\n")
cat(tfidf_summary_text, "\n")
cat(sprintf("Words: %d / %d (Compression Ratio: %.1f%% reduction)\n\n",
            str_count(tfidf_summary_text, "\\S+"), sum(sentences_df$raw_words),
            100 * (1 - str_count(tfidf_summary_text, "\\S+") / sum(sentences_df$raw_words))))

# Corpus keyword & bigram analysis
term_totals <- data.frame(
  term = colnames(tfidf_mat),
  total_tfidf = colSums(tfidf_mat),
  raw_freq = colSums(as.matrix(dtm_tf)),
  stringsAsFactors = FALSE
) %>%
  arrange(desc(total_tfidf))

write.csv(term_totals, file.path(out_dir, "q3_top_tfidf_keywords.csv"), row.names = FALSE)

# Bigrams extraction
bigrams_df <- sentences_df %>%
  tidytext::unnest_tokens(bigram, clean_text, token = "ngrams", n = 2) %>%
  filter(!is.na(bigram)) %>%
  count(bigram, sort = TRUE)
write.csv(bigrams_df, file.path(out_dir, "q3_top_bigrams.csv"), row.names = FALSE)

# =============================================================================
# VISUALISATIONS FOR QUESTION 3.3 (3 FULLY LABELLED CHARTS)
# =============================================================================
# Visualisation 1: Sentence Ranking Comparison (TextRank vs TF-IDF Across Article Flow)
score_long <- sentences_df %>%
  select(sentence_id, selection_status, TextRank = textrank_norm, `TF-IDF Score` = tfidf_norm) %>%
  pivot_longer(cols = c("TextRank", "TF-IDF Score"), names_to = "Method", values_to = "Score")

p3_1 <- ggplot(score_long, aes(x = factor(sentence_id), y = Score, fill = selection_status)) +
  geom_col(width = 0.72) +
  geom_text(aes(label = sprintf("%.2f", Score)), vjust = -0.35, size = 2.6, fontface = "bold", color = "#1E293B") +
  facet_wrap(~ Method, ncol = 1) +
  scale_fill_manual(values = c(
    "Selected by Both Methods" = "#D97706",
    "TextRank Only"            = "#0D9488",
    "TF-IDF Only"              = "#2563EB",
    "Not Selected"             = "#94A3B8"
  ), name = "Summary Selection") +
  ylim(0, 1.18) +
  labs(
    title = "Text Summarisation Comparison: Sentence Importance Across Article Narrative",
    subtitle = "Top 5 sentences selected independently by Graph-Based TextRank and TF-IDF Centroid Scoring (Eurac Research Article)",
    x = "Sentence Position Index in Original Article (S1 to S22)",
    y = "Normalised Importance Score (0 to 1)"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 13),
    plot.subtitle = element_text(color = "#475569", size = 9.8),
    strip.text = element_text(face = "bold", color = "#1B365D", size = 11),
    strip.background = element_rect(fill = "#F1F5F9", color = NA),
    legend.position = "top"
  )

ggsave(file.path(out_dir, "q3_sentence_ranking_comparison.png"), p3_1, width = 12, height = 7.0, dpi = 180, bg = "white")

# Visualisation 2: Top 15 Keywords by Cumulative TF-IDF Weight & Top 10 Domain Bigrams
p_kw <- term_totals %>%
  slice_head(n = 15) %>%
  mutate(term = reorder(term, total_tfidf)) %>%
  ggplot(aes(x = term, y = total_tfidf, fill = total_tfidf)) +
  geom_col(show.legend = FALSE, width = 0.7) +
  geom_text(aes(label = sprintf("%.2f (n=%d)", total_tfidf, raw_freq)), hjust = -0.06, size = 3.1, fontface = "bold", color = "#1E293B") +
  coord_flip(ylim = c(0, max(term_totals$total_tfidf) * 1.32)) +
  scale_fill_gradient(low = "#3B82F6", high = "#1B365D") +
  labs(
    title = "A. Top 15 Keywords by Cumulative TF-IDF Weight",
    subtitle = "Domain terms capturing drought, climate impacts & NLP reasoning",
    x = NULL, y = "Cumulative Sentence TF-IDF Score"
  ) +
  theme_minimal(base_size = 10.5) +
  theme(plot.title = element_text(face = "bold", color = "#1B365D"), panel.grid.major.y = element_blank())

p_bg <- bigrams_df %>%
  slice_head(n = 12) %>%
  mutate(bigram = reorder(bigram, n)) %>%
  ggplot(aes(x = bigram, y = n)) +
  geom_col(fill = "#0D9488", width = 0.68) +
  geom_text(aes(label = sprintf("%d hits", n)), hjust = -0.1, size = 3.2, fontface = "bold", color = "#1E293B") +
  coord_flip(ylim = c(0, max(bigrams_df$n) * 1.25)) +
  labs(
    title = "B. Top 12 Most Frequent Domain Bigrams",
    subtitle = "Multi-word expressions extracted after stopword denoising",
    x = NULL, y = "Frequency Across Article"
  ) +
  theme_minimal(base_size = 10.5) +
  theme(plot.title = element_text(face = "bold", color = "#1B365D"), panel.grid.major.y = element_blank())

g_kw <- gridExtra::arrangeGrob(
  p_kw, p_bg, ncol = 2,
  top = grid::textGrob("Semantic Keyphrase & Co-Occurrence Analysis ('Can language become environmental data?')",
                       gp = grid::gpar(fontsize = 13, fontface = "bold", col = "#1B365D"))
)
ggsave(file.path(out_dir, "q3_keyword_tfidf_bigrams.png"), g_kw, width = 13, height = 6.2, dpi = 180, bg = "white")

# Visualisation 3: Compression Ratio & Information Density Metrics
orig_words <- sum(sentences_df$raw_words)
orig_sents <- nrow(sentences_df)
orig_chars <- sum(sentences_df$raw_chars)
tr_words   <- str_count(tr_summary_text, "\\S+")
tf_words   <- str_count(tfidf_summary_text, "\\S+")
tr_chars   <- nchar(tr_summary_text)
tf_chars   <- nchar(tfidf_summary_text)

# Unique top-15 keyword coverage rate inside each summary
top15_words <- term_totals$term[1:15]
cov_orig <- 100
cov_tr   <- round(100 * mean(sapply(top15_words, function(w) grepl(w, tolower(tr_summary_text)))), 1)
cov_tf   <- round(100 * mean(sapply(top15_words, function(w) grepl(w, tolower(tfidf_summary_text)))), 1)

comp_df <- data.frame(
  Version = factor(c("Original Article\n(Full Text)", "Method 1:\nTextRank Summary", "Method 2:\nTF-IDF Summary"),
                   levels = c("Original Article\n(Full Text)", "Method 1:\nTextRank Summary", "Method 2:\nTF-IDF Summary")),
  Word_Count = c(orig_words, tr_words, tf_words),
  Word_Reduction_Pct = c(0, round(100 * (1 - tr_words / orig_words), 1), round(100 * (1 - tf_words / orig_words), 1)),
  Top15_Keyword_Coverage = c(cov_orig, cov_tr, cov_tf)
)

p3_3a <- ggplot(comp_df, aes(x = Version, y = Word_Count, fill = Version)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = ifelse(Word_Reduction_Pct == 0,
                               sprintf("%d words\n(Baseline)", Word_Count),
                               sprintf("%d words\n(-%.1f%% size)", Word_Count, Word_Reduction_Pct))),
            vjust = -0.25, size = 3.4, fontface = "bold", color = "#1E293B") +
  scale_fill_manual(values = c("#1B365D", "#0D9488", "#D97706")) +
  ylim(0, orig_words * 1.22) +
  labs(title = "A. Word Volume & Text Compression Ratio", x = NULL, y = "Word Count") +
  theme_minimal(base_size = 10.5) +
  theme(plot.title = element_text(face = "bold", color = "#1B365D"), panel.grid.major.x = element_blank())

p3_3b <- ggplot(comp_df, aes(x = Version, y = Top15_Keyword_Coverage, fill = Version)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.1f%% coverage\n(of Top-15 TF-IDF terms)", Top15_Keyword_Coverage)),
            vjust = -0.25, size = 3.3, fontface = "bold", color = "#1E293B") +
  scale_fill_manual(values = c("#1B365D", "#0D9488", "#D97706")) +
  ylim(0, 122) +
  labs(title = "B. Semantic Information Retention Rate", x = NULL, y = "Top-15 Keyword Retention (%)") +
  theme_minimal(base_size = 10.5) +
  theme(plot.title = element_text(face = "bold", color = "#1B365D"), panel.grid.major.x = element_blank())

g_comp <- gridExtra::arrangeGrob(
  p3_3a, p3_3b, ncol = 2,
  top = grid::textGrob("Summarisation Evaluation: Length Compression vs Key-Concept Retention",
                       gp = grid::gpar(fontsize = 13, fontface = "bold", col = "#1B365D"))
)
ggsave(file.path(out_dir, "q3_compression_readability_metrics.png"), g_comp, width = 12, height = 5.4, dpi = 180, bg = "white")

cat("Saved all 3 Question 3 visualisations in outputs/.\n")


WebText API Extraction & Preprocessing Verification:
Source: https://www.eurac.edu/en/blogs/connecting-the-dots/can-language-become-environmental-data (HTTP 200)
Paragraphs: 5 | Sentences: 19 | Raw words: 511 -> Clean tokens: 313 (38.7% noise removed)

=== TOP TF-IDF KEYWORDS ===
"term","total_tfidf","raw_freq"
"environmental",0.901064032651232,7
"access",0.849585502688717,1
"kind",0.849585502688717,1
"research",0.849585502688717,1
"information",0.79336318697528,6
"specific",0.772350456989743,2
"drought",0.77149098353573,8
"language",0.769403495705287,7
"data",0.730956222886622,5
"water",0.680007137177335,4
"valuable",0.610721412784264,2
"become",0.606846787634798,1
"dryalps",0.606846787634798,1
"investigate",0.606846787634798,1
"project",0.606846787634798,1
"impacts",0.601096063429508,9
"society",0.551935394441394,2
"time",0.547579385732967,3
"systems",0.519650775835011,4
"texts",0.51554404975295,2
"measurements",0.487189127016538,2
"affects",0.471991945938176,1
"contain",0.4719919459

## 3.3 Dual Extractive Summarisation Findings & Visualisations
- **Method 1 (Graph-Based TextRank / PageRank via `igraph`)**: Selects Sentences **4, 6, 8, 14, 19** (`177 / 511` words, **65.4% compression**, **46.7% top-15 TF-IDF keyword coverage**).
- **Method 2 (TF-IDF Weighted Sentence Scoring)**: Selects Sentences **6, 7, 10, 17, 18** (`84 / 511` words, **83.6% compression**, **93.3% top-15 TF-IDF keyword coverage**).
- **Consensus Sentence**: Sentence 6 (*'Together, these texts contain valuable information about how drought affects society and the environment.'*) appears in both summaries.

| term | total_tfidf | raw_freq |
| --- | --- | --- |
| environmental | 0.901064032651232 | 7 |
| access | 0.849585502688717 | 1 |
| kind | 0.849585502688717 | 1 |
| research | 0.849585502688717 | 1 |
| information | 0.79336318697528 | 6 |
| specific | 0.772350456989743 | 2 |
| drought | 0.77149098353573 | 8 |
| language | 0.769403495705287 | 7 |
| data | 0.730956222886622 | 5 |
| water | 0.680007137177335 | 4 |

![Sentence Ranking Comparison](../outputs/q3_sentence_ranking_comparison.png)

![Top TF-IDF Keywords and Bigrams](../outputs/q3_keyword_tfidf_bigrams.png)

![Summarisation Compression vs Coverage](../outputs/q3_compression_readability_metrics.png)


---


# ITBDA4-14 Project — Question 4: End-to-End ETL Pipeline in R & PostgreSQL 17
**Use Case:** Scraping, Regex Parsing, Cleaning, and Analysing the Eduvos Academic Programme Portfolio (`https://www.eduvos.com/qualifications/`)

## 4.1 Raw Data Extraction & Anomaly Audit (`N = 68` Programmes)
Verified `robots.txt` (`Allow: /`) and parsed `<article class='qf-card'>` records with `rvest` and `stringr` regex rules, capturing real-world web imperfections prior to cleaning:

| anomaly_category | affected_records | total_records | pct_affected |
| --- | --- | --- | --- |
| Missing data-learning-mode attribute (NA) | 1 | 68 | 1.5 |
| Missing explicit NQF level (Access / Pre-Degree / Corporate) | 20 | 68 | 29.4 |
| Missing explicit duration in years (Access / Corporate) | 15 | 68 | 22.1 |
| Truncated descriptions ending with ellipsis (...) | 67 | 68 | 98.5 |
| Unparsed date prefix ('Start date: ...') | 68 | 68 | 100 |
| Unicode en/em dashes requiring ASCII normalisation | 6 | 68 | 8.8 |
| Boilerplate duplicate card descriptions (repeated text) | 10 | 68 | 14.7 |
| Misclassified Pre-Degree IT/Science under Humanities & Arts | 2 | 68 | 2.9 |
| Relative URL paths missing domain scheme/host | 68 | 68 | 100 |


In [1]:
# ==============================================================================
# ITBDA4-14 Question 4: ETL Manipulating Using R & PostgreSQL
# Source: Eduvos Qualifications Catalogue (https://www.eduvos.com/qualifications/)
# ==============================================================================

suppressPackageStartupMessages({
  library(rvest)
  library(xml2)
  library(httr)
  library(stringr)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(scales)
  library(DBI)
  library(RPostgres)
})

base_dir <- normalizePath(if (dir.exists("data")) "." else "..", winslash = "/")
data_dir <- file.path(base_dir, "data")
out_dir  <- file.path(base_dir, "outputs")
sql_dir  <- file.path(base_dir, "src", "sql")
dir.create(sql_dir, recursive = TRUE, showWarnings = FALSE)

cache_html <- file.path(data_dir, "eduvos_qualifications_cached.html")
target_url <- "https://www.eduvos.com/qualifications/"

# ------------------------------------------------------------------------------
# STEP 4.1: EXTRACT - WEB SCRAPING & REGEX PARSING (RAW UNSTRUCTURED DATASET)
# ------------------------------------------------------------------------------
if (!file.exists(cache_html)) {
  resp <- GET(
    target_url,
    user_agent("Mozilla/5.0 (Windows NT 10.0; Win64; x64) ITBDA4-14-ETL-Research/1.0"),
    timeout(30)
  )
  stopifnot(status_code(resp) == 200)
  html_raw <- content(resp, as = "text", encoding = "UTF-8")
  writeLines(html_raw, cache_html, useBytes = TRUE)
} else {
  html_raw <- paste(readLines(cache_html, encoding = "UTF-8", warn = FALSE), collapse = "\n")
}

pg <- read_html(html_raw)
cards <- pg %>% html_elements("article.qf-card")
cat(sprintf("Extracted %d raw <article class='qf-card'> records from Eduvos Qualifications.\n", length(cards)))

extract_text_safe <- function(node, selector) {
  el <- html_element(node, selector)
  if (inherits(el, "xml_missing")) return(NA_character_)
  val <- str_squish(html_text2(el))
  if (val == "") return(NA_character_)
  val
}

raw_onclick <- vapply(cards, function(c) {
  val <- html_attr(c, "onclick")
  if (is.na(val)) "" else val
}, character(1))

# Regex extraction of URL slug inside window.open('...')
url_slug <- str_match(raw_onclick, "window\\.open\\('([^']+)'")[, 2]
href_attr <- vapply(cards, function(c) {
  el <- html_element(c, ".qf-card-title a")
  if (inherits(el, "xml_missing")) NA_character_ else html_attr(el, "href")
}, character(1))
rel_url <- ifelse(!is.na(href_attr) & nzchar(href_attr), href_attr, url_slug)

learning_modes_attr <- vapply(cards, function(c) {
  val <- html_attr(c, "data-learning-mode")
  if (is.na(val) || str_trim(val) == "") NA_character_ else str_trim(val)
}, character(1))

title_raw <- vapply(cards, extract_text_safe, character(1), selector = ".qf-card-title a")
type_raw  <- vapply(cards, extract_text_safe, character(1), selector = ".qf-type-corner")
fac_raw   <- vapply(cards, extract_text_safe, character(1), selector = ".qf-card-faculty")
desc_raw  <- vapply(cards, extract_text_safe, character(1), selector = ".qf-card-description")
start_raw <- vapply(cards, extract_text_safe, character(1), selector = ".qf-start-date")

# Regex inference of raw numerical attributes (intentionally preserving NAs where card metadata is absent)
infer_nqf_raw <- function(t_type, t_title) {
  case_when(
    str_detect(t_type, regex("^Higher Certificate$", ignore_case = TRUE)) ~ 5L,
    str_detect(t_type, regex("^Degree$", ignore_case = TRUE)) &
      str_detect(t_title, regex("Bachelor of Laws|\\bLLB\\b", ignore_case = TRUE)) ~ 8L,
    str_detect(t_type, regex("^Degree$", ignore_case = TRUE)) ~ 7L,
    str_detect(t_type, regex("Postgraduate|Honours", ignore_case = TRUE)) ~ 8L,
    str_detect(t_type, regex("^Masters$", ignore_case = TRUE)) ~ 9L,
    str_detect(t_type, regex("^Doctorates$", ignore_case = TRUE)) ~ 10L,
    TRUE ~ NA_integer_  # Access, Pre-Degree, Corporate Training have no explicit NQF tier
  )
}

infer_duration_raw <- function(t_type, t_title) {
  case_when(
    str_detect(t_type, regex("^Higher Certificate$|Postgraduate|Honours", ignore_case = TRUE)) ~ 1.0,
    str_detect(t_type, regex("^Degree$", ignore_case = TRUE)) &
      str_detect(t_title, regex("Bachelor of Laws|\\bLLB\\b", ignore_case = TRUE)) ~ 4.0,
    str_detect(t_type, regex("^Degree$", ignore_case = TRUE)) ~ 3.0,
    str_detect(t_type, regex("^Masters$", ignore_case = TRUE)) ~ 2.0,
    str_detect(t_type, regex("^Doctorates$", ignore_case = TRUE)) ~ 3.0,
    str_detect(t_type, regex("^Pre-Degree$", ignore_case = TRUE)) ~ 1.0,
    TRUE ~ NA_real_     # Access & Corporate Training lack explicit duration on listing cards
  )
}

raw_df <- data.frame(
  programme_id           = sprintf("EDV-%03d", seq_along(cards)),
  programme_title_raw    = title_raw,
  faculty_raw            = fac_raw,
  qualification_type_raw = type_raw,
  learning_modes_raw     = learning_modes_attr,
  start_date_raw         = start_raw,
  description_raw        = desc_raw,
  programme_url_raw      = rel_url,
  nqf_level_raw          = infer_nqf_raw(type_raw, title_raw),
  min_duration_years_raw = infer_duration_raw(type_raw, title_raw),
  delivery_mode_count_raw = ifelse(
    is.na(learning_modes_attr),
    NA_integer_,
    str_count(learning_modes_attr, ",") + 1L
  ),
  desc_word_count_raw    = str_count(desc_raw, "\\S+"),
  desc_char_count_raw    = nchar(desc_raw),
  stringsAsFactors       = FALSE
)

write.csv(raw_df, file.path(data_dir, "q4_eduvos_programmes_raw.csv"), row.names = FALSE, na = "NA")

# Quantify real-world anomalies / imperfections prior to cleaning
audit_df <- data.frame(
  anomaly_category = c(
    "Missing data-learning-mode attribute (NA)",
    "Missing explicit NQF level (Access / Pre-Degree / Corporate)",
    "Missing explicit duration in years (Access / Corporate)",
    "Truncated descriptions ending with ellipsis (...)",
    "Unparsed date prefix ('Start date: ...')",
    "Unicode en/em dashes requiring ASCII normalisation",
    "Boilerplate duplicate card descriptions (repeated text)",
    "Misclassified Pre-Degree IT/Science under Humanities & Arts",
    "Relative URL paths missing domain scheme/host"
  ),
  affected_records = c(
    sum(is.na(raw_df$learning_modes_raw)),
    sum(is.na(raw_df$nqf_level_raw)),
    sum(is.na(raw_df$min_duration_years_raw)),
    sum(str_detect(raw_df$description_raw, "\\.\\.\\.$"), na.rm = TRUE),
    sum(str_detect(raw_df$start_date_raw, "^Start date:"), na.rm = TRUE),
    sum(str_detect(paste(raw_df$programme_title_raw, raw_df$description_raw), "\u2013|\u2014"), na.rm = TRUE),
    sum(duplicated(raw_df$description_raw) | duplicated(raw_df$description_raw, fromLast = TRUE)),
    sum(raw_df$faculty_raw == "Humanities and Arts" &
          str_detect(raw_df$programme_title_raw, regex(":\\s*(Information Technology|Science)$", ignore_case = TRUE))),
    sum(str_detect(raw_df$programme_url_raw, "^/"), na.rm = TRUE)
  ),
  total_records = nrow(raw_df),
  stringsAsFactors = FALSE
)
audit_df$pct_affected <- round(100 * audit_df$affected_records / audit_df$total_records, 1)
write.csv(audit_df, file.path(out_dir, "q4_data_quality_audit.csv"), row.names = FALSE)

cat("\n=== 4.1 RAW DATA QUALITY AUDIT (N =", nrow(raw_df), ") ===\n")
print(audit_df)

# ------------------------------------------------------------------------------
# STEP 4.2: TRANSFORM - CLEANING, DENOISING & MISSING-VALUE IMPUTATION
# ------------------------------------------------------------------------------
clean_df <- raw_df %>%
  mutate(
    # Standardise titles & Unicode dashes
    programme_title_clean = str_squish(str_replace_all(programme_title_raw, "[\u2013\u2014]", "-")),
    # Correct domain misclassifications observed on live site
    faculty_corrected = (
      faculty_raw == "Humanities and Arts" &
        str_detect(programme_title_raw, regex(":\\s*(Information Technology|Science)$", ignore_case = TRUE))
    ),
    faculty_clean = case_when(
      faculty_raw == "Humanities and Arts" &
        str_detect(programme_title_raw, regex(":\\s*Information Technology$", ignore_case = TRUE)) ~ "Technology",
      faculty_raw == "Humanities and Arts" &
        str_detect(programme_title_raw, regex(":\\s*Science$", ignore_case = TRUE)) ~ "Applied Sciences",
      TRUE ~ str_squish(faculty_raw)
    ),
    qualification_type = str_squish(qualification_type_raw),
    study_level_group = case_when(
      qualification_type %in% c("Access", "Pre-Degree") ~ "Foundational",
      qualification_type %in% c("Higher Certificate", "Degree") ~ "Undergraduate",
      qualification_type %in% c("Honours", "Postgraduate Certificates & Diplomas", "Masters", "Doctorates") ~ "Postgraduate",
      TRUE ~ "Corporate & Short Courses"
    ),
    # Impute single missing learning mode with faculty/degree modal value
    learning_mode_imputed = is.na(learning_modes_raw),
    learning_modes_clean = ifelse(
      is.na(learning_modes_raw),
      "Campus Full Time, Online Full Time, Online Part Time",
      str_squish(learning_modes_raw)
    ),
    delivery_mode_count = str_count(learning_modes_clean, ",") + 1L,
    offers_online = str_detect(learning_modes_clean, regex("Online|Remote|Distance", ignore_case = TRUE)),
    offers_campus = str_detect(learning_modes_clean, regex("Campus", ignore_case = TRUE)),
    # Parse raw 'Start date: DD Month YYYY' into ISO Date
    start_date_str = str_trim(str_remove(start_date_raw, "^Start date:\\s*")),
    intake_date_iso = as.character(as.Date(start_date_str, format = "%d %B %Y")),
    intake_cohort = format(as.Date(start_date_str, format = "%d %B %Y"), "%Y-%m"),
    # Denoise description text (remove truncation ellipsis, Unicode dashes)
    was_truncated = str_detect(description_raw, "\\.\\.\\.$"),
    description_clean = description_raw %>%
      str_replace_all("[\u2013\u2014]", "-") %>%
      str_remove("\\.\\.\\.$") %>%
      str_squish(),
    desc_word_count = str_count(description_clean, "\\S+"),
    desc_char_count = nchar(description_clean),
    programme_url_full = ifelse(
      str_starts(programme_url_raw, "http"),
      programme_url_raw,
      paste0("https://www.eduvos.com", programme_url_raw)
    ),
    # Deterministic rule-based imputation for NQF level and duration
    nqf_imputed = is.na(nqf_level_raw),
    nqf_level = case_when(
      !is.na(nqf_level_raw) ~ nqf_level_raw,
      qualification_type %in% c("Access", "Pre-Degree") ~ 5L,
      qualification_type == "Corporate Training" &
        str_detect(programme_title_clean, regex("National Certificate", ignore_case = TRUE)) ~ 5L,
      qualification_type == "Corporate Training" ~ 6L,
      TRUE ~ 5L
    ),
    duration_imputed = is.na(min_duration_years_raw),
    min_duration_years = case_when(
      !is.na(min_duration_years_raw) ~ min_duration_years_raw,
      qualification_type == "Access" ~ 1.0,
      qualification_type == "Corporate Training" ~ 0.25,
      TRUE ~ 1.0
    ),
    estimated_saqa_credits = as.integer(round(min_duration_years * 120))
  ) %>%
  select(
    programme_id, programme_title_clean, faculty_raw, faculty_clean, faculty_corrected,
    qualification_type, study_level_group, nqf_level, nqf_imputed,
    min_duration_years, duration_imputed, estimated_saqa_credits,
    learning_modes_clean, learning_mode_imputed, delivery_mode_count,
    offers_online, offers_campus, intake_date_iso, intake_cohort,
    description_clean, was_truncated, desc_word_count, desc_char_count,
    programme_url_full
  )

cat("\n=== 4.2 CLEANED DATASET VERIFICATION ===\n")
cat("Remaining NAs across cleaned columns:", sum(is.na(clean_df)), "\n")
cat("Faculties corrected:", sum(clean_df$faculty_corrected), "\n")
cat("Learning mode imputed:", sum(clean_df$learning_mode_imputed), "\n")
cat("NQF levels imputed:", sum(clean_df$nqf_imputed), "\n")
cat("Durations imputed:", sum(clean_df$duration_imputed), "\n")

write.csv(clean_df, file.path(data_dir, "q4_eduvos_programmes_cleaned.csv"), row.names = FALSE)
write.csv(clean_df, file.path(out_dir,  "q4_eduvos_programmes_cleaned.csv"), row.names = FALSE)

# ------------------------------------------------------------------------------
# STEP 4.3: LOAD INTO POSTGRESQL 17 & EXECUTE SQL ANALYTICAL INVESTIGATIONS
# ------------------------------------------------------------------------------
con <- dbConnect(
  RPostgres::Postgres(),
  host     = "127.0.0.1",
  port     = 5433L,
  user     = "postgres",
  password = "",
  dbname   = "itbda4_etl_db"
)
on.exit(dbDisconnect(con), add = TRUE)

dbExecute(con, "DROP TABLE IF EXISTS eduvos_programmes;")
dbExecute(con, "
CREATE TABLE eduvos_programmes (
  programme_id           VARCHAR(16) PRIMARY KEY,
  programme_title_clean  TEXT NOT NULL,
  faculty_raw            VARCHAR(80) NOT NULL,
  faculty_clean          VARCHAR(80) NOT NULL,
  faculty_corrected      BOOLEAN NOT NULL,
  qualification_type     VARCHAR(80) NOT NULL,
  study_level_group      VARCHAR(40) NOT NULL,
  nqf_level              INTEGER NOT NULL CHECK (nqf_level BETWEEN 5 AND 10),
  nqf_imputed            BOOLEAN NOT NULL,
  min_duration_years     NUMERIC(4,2) NOT NULL CHECK (min_duration_years > 0),
  duration_imputed       BOOLEAN NOT NULL,
  estimated_saqa_credits INTEGER NOT NULL CHECK (estimated_saqa_credits > 0),
  learning_modes_clean   TEXT NOT NULL,
  learning_mode_imputed  BOOLEAN NOT NULL,
  delivery_mode_count    INTEGER NOT NULL,
  offers_online          BOOLEAN NOT NULL,
  offers_campus          BOOLEAN NOT NULL,
  intake_date_iso        DATE NOT NULL,
  intake_cohort          VARCHAR(10) NOT NULL,
  description_clean      TEXT NOT NULL,
  was_truncated          BOOLEAN NOT NULL,
  desc_word_count        INTEGER NOT NULL,
  desc_char_count        INTEGER NOT NULL,
  programme_url_full     TEXT NOT NULL
);")

# Prepare types cleanly for RPostgres append
pg_df <- clean_df %>%
  mutate(intake_date_iso = as.Date(intake_date_iso))

dbWriteTable(con, "eduvos_programmes", pg_df, append = TRUE, row.names = FALSE)
row_check <- dbGetQuery(con, "SELECT COUNT(*) AS loaded_rows FROM eduvos_programmes;")
cat(sprintf("\nPostgreSQL table 'eduvos_programmes' populated successfully (%d rows).\n", as.integer(row_check$loaded_rows)))

coerce_pg_types <- function(df) {
  df %>% mutate(across(where(~ inherits(.x, "integer64")), as.numeric))
}

# SQL Analytical Investigation 1: Faculty Portfolio Composition, NQF Depth & Modality Mix
sql_q1 <- "
SELECT
  faculty_clean AS faculty,
  COUNT(*) AS total_programmes,
  SUM(CASE WHEN study_level_group = 'Foundational' THEN 1 ELSE 0 END) AS foundational_cnt,
  SUM(CASE WHEN study_level_group = 'Undergraduate' THEN 1 ELSE 0 END) AS undergrad_cnt,
  SUM(CASE WHEN study_level_group = 'Postgraduate' THEN 1 ELSE 0 END) AS postgrad_cnt,
  SUM(CASE WHEN study_level_group = 'Corporate & Short Courses' THEN 1 ELSE 0 END) AS corporate_cnt,
  ROUND(AVG(nqf_level)::numeric, 2) AS avg_nqf_level,
  ROUND(AVG(estimated_saqa_credits)::numeric, 1) AS avg_saqa_credits,
  ROUND(AVG(delivery_mode_count)::numeric, 2) AS avg_delivery_modes,
  ROUND(100.0 * SUM(CASE WHEN offers_online THEN 1 ELSE 0 END) / COUNT(*), 1) AS online_pct
FROM eduvos_programmes
GROUP BY faculty_clean
ORDER BY total_programmes DESC, avg_nqf_level DESC;
"

# SQL Analytical Investigation 2: Qualification Tier & Intake Date Cohort Structure
sql_q2 <- "
SELECT
  qualification_type,
  study_level_group,
  intake_date_iso,
  ROUND(AVG(nqf_level)::numeric, 1) AS avg_nqf,
  ROUND(AVG(min_duration_years)::numeric, 2) AS avg_years,
  COUNT(*) AS programme_count,
  SUM(estimated_saqa_credits) AS total_saqa_credits,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_catalogue,
  SUM(CASE WHEN nqf_imputed THEN 1 ELSE 0 END) AS nqf_imputed_cnt
FROM eduvos_programmes
GROUP BY qualification_type, study_level_group, intake_date_iso
ORDER BY intake_date_iso ASC, avg_nqf DESC, programme_count DESC;
"

# SQL Analytical Investigation 3: Window Function Ranking Highest-NQF Flagships per Faculty
sql_q3 <- "
WITH ranked_programmes AS (
  SELECT
    faculty_clean,
    programme_id,
    programme_title_clean,
    qualification_type,
    nqf_level,
    estimated_saqa_credits,
    delivery_mode_count,
    DENSE_RANK() OVER (
      PARTITION BY faculty_clean
      ORDER BY nqf_level DESC, estimated_saqa_credits DESC, programme_id ASC
    ) AS faculty_nqf_rank
  FROM eduvos_programmes
)
SELECT
  faculty_clean AS faculty,
  faculty_nqf_rank,
  programme_id,
  programme_title_clean,
  qualification_type,
  nqf_level,
  estimated_saqa_credits,
  delivery_mode_count
FROM ranked_programmes
WHERE faculty_nqf_rank <= 2
ORDER BY faculty_clean ASC, faculty_nqf_rank ASC, programme_id ASC;
"

res_q1 <- coerce_pg_types(dbGetQuery(con, sql_q1))
res_q2 <- coerce_pg_types(dbGetQuery(con, sql_q2))
res_q3 <- coerce_pg_types(dbGetQuery(con, sql_q3))

write.csv(res_q1, file.path(out_dir, "q4_sql_query1_faculty_summary.csv"), row.names = FALSE)
write.csv(res_q2, file.path(out_dir, "q4_sql_query2_tier_intake_matrix.csv"), row.names = FALSE)
write.csv(res_q3, file.path(out_dir, "q4_sql_query3_faculty_top_programmes.csv"), row.names = FALSE)

cat("\n=== SQL INVESTIGATION 1: FACULTY SUMMARY ===\n")
print(res_q1)
cat("\n=== SQL INVESTIGATION 2: QUALIFICATION TIER & INTAKE MATRIX ===\n")
print(res_q2)
cat("\n=== SQL INVESTIGATION 3: TOP RANKED PROGRAMMES PER FACULTY (SAMPLE) ===\n")
print(head(res_q3, 12))

# ------------------------------------------------------------------------------
# VISUALISATION OF POSTGRESQL ANALYTICAL RESULTS (2-PANEL COMPOSITE)
# ------------------------------------------------------------------------------
tier_counts <- clean_df %>%
  count(faculty_clean, study_level_group) %>%
  left_join(res_q1 %>% select(faculty, total_programmes), by = c("faculty_clean" = "faculty")) %>%
  mutate(
    study_level_group = factor(
      study_level_group,
      levels = c("Foundational", "Undergraduate", "Postgraduate", "Corporate & Short Courses")
    ),
    faculty_ord = reorder(faculty_clean, total_programmes)
  )

p1 <- ggplot(tier_counts, aes(x = faculty_ord, y = n, fill = study_level_group)) +
  geom_col(width = 0.72, color = "white", linewidth = 0.35) +
  geom_text(
    aes(label = n),
    position = position_stack(vjust = 0.5),
    color = "white", fontface = "bold", size = 3.2
  ) +
  coord_flip() +
  scale_fill_manual(
    values = c(
      "Foundational"              = "#64748B",
      "Undergraduate"             = "#1B365D",
      "Postgraduate"              = "#0EA5E9",
      "Corporate & Short Courses" = "#F59E0B"
    ),
    name = "Academic Tier"
  ) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.14))) +
  labs(
    title = "A. Programme Portfolio Across Faculties (PostgreSQL Cleaned Table)",
    subtitle = "Stacked programme counts after correcting 2 Pre-Degree faculty misclassifications",
    x = NULL,
    y = "Number of Programmes"
  ) +
  theme_minimal(base_size = 10.5) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 11.2),
    plot.subtitle = element_text(color = "#475569", size = 8.8),
    legend.position = "bottom",
    legend.title = element_text(face = "bold", size = 9),
    plot.margin = margin(10, 14, 10, 10),
    panel.grid.major.y = element_blank()
  )

res_q1_plot <- res_q1 %>%
  mutate(
    faculty_ord = reorder(faculty, avg_nqf_level),
    label_txt = sprintf("NQF %.2f  |  %.0f credits", avg_nqf_level, avg_saqa_credits)
  )

p2 <- ggplot(res_q1_plot, aes(x = faculty_ord, y = avg_nqf_level)) +
  geom_segment(aes(xend = faculty_ord, y = 4.5, yend = avg_nqf_level), color = "#CBD5E1", linewidth = 2.2) +
  geom_point(aes(size = avg_saqa_credits), color = "#1D4ED8", alpha = 0.9) +
  geom_text(aes(label = label_txt), nudge_y = 0.26, hjust = 0, size = 3.2, fontface = "bold", color = "#1E293B") +
  coord_flip(ylim = c(4.5, 10.2)) +
  scale_size_continuous(range = c(5, 11), name = "Mean SAQA Credits") +
  labs(
    title = "B. Mean NQF Depth & SAQA Credit Load by Faculty (SQL Aggregation)",
    subtitle = "PostgreSQL 17 GROUP BY query over 68 cleaned Eduvos qualification records",
    x = NULL,
    y = "Mean NQF Level (5 to 10 Scale)"
  ) +
  theme_minimal(base_size = 10.5) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 11.2),
    plot.subtitle = element_text(color = "#475569", size = 8.8),
    legend.position = "bottom",
    legend.title = element_text(face = "bold", size = 9),
    plot.margin = margin(10, 14, 10, 14),
    panel.grid.major.y = element_blank()
  )

# Save two side-by-side panels cleanly via gridExtra or grid viewport
png(file.path(out_dir, "q4_etl_postgresql_analysis.png"), width = 2400, height = 1120, res = 175, bg = "white")
grid::grid.newpage()
grid::pushViewport(grid::viewport(layout = grid::grid.layout(1, 2)))
print(p1, vp = grid::viewport(layout.pos.row = 1, layout.pos.col = 1))
print(p2, vp = grid::viewport(layout.pos.row = 1, layout.pos.col = 2))
dev.off()

cat("\nSaved outputs/q4_etl_postgresql_analysis.png and all Q4 CSV deliverables.\n")


PostgreSQL table 'eduvos_programmes' populated successfully (68 rows). Remaining NAs: 0.


## 4.3 PostgreSQL 17 Relational Analytics (`src/sql/q4_analytics.sql`)


In [2]:
-- ============================================================================
-- ITBDA4-14 Question 4.3: PostgreSQL Relational Analytics Script
-- Database: itbda4_etl_db | Table: eduvos_programmes (68 cleaned records)
-- ============================================================================

-- Verify PostgreSQL version and loaded table row count
SELECT version();
SELECT COUNT(*) AS total_loaded_programmes FROM eduvos_programmes;

-- Investigation 1: Faculty Portfolio Composition, NQF Depth, and Online Accessibility
SELECT
    faculty_clean AS faculty,
    COUNT(*) AS total_programmes,
    SUM(CASE WHEN study_level_group = 'Foundational' THEN 1 ELSE 0 END) AS foundational_cnt,
    SUM(CASE WHEN study_level_group = 'Undergraduate' THEN 1 ELSE 0 END) AS undergrad_cnt,
    SUM(CASE WHEN study_level_group = 'Postgraduate' THEN 1 ELSE 0 END) AS postgrad_cnt,
    SUM(CASE WHEN study_level_group = 'Corporate & Short Courses' THEN 1 ELSE 0 END) AS corporate_cnt,
    ROUND(AVG(nqf_level)::numeric, 2) AS avg_nqf_level,
    ROUND(AVG(estimated_saqa_credits)::numeric, 1) AS avg_saqa_credits,
    ROUND(AVG(delivery_mode_count)::numeric, 2) AS avg_delivery_modes,
    ROUND(100.0 * SUM(CASE WHEN offers_online THEN 1 ELSE 0 END) / COUNT(*), 1) AS online_pct
FROM eduvos_programmes
GROUP BY faculty_clean
ORDER BY total_programmes DESC, avg_nqf_level DESC;

-- Investigation 2: Qualification Tier, Intake Schedule, and Imputation Impact Analysis
SELECT
    qualification_type,
    study_level_group,
    intake_date_iso,
    ROUND(AVG(nqf_level)::numeric, 1) AS avg_nqf,
    ROUND(AVG(min_duration_years)::numeric, 2) AS avg_years,
    COUNT(*) AS programme_count,
    SUM(estimated_saqa_credits) AS total_saqa_credits,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_catalogue,
    SUM(CASE WHEN nqf_imputed THEN 1 ELSE 0 END) AS nqf_imputed_cnt
FROM eduvos_programmes
GROUP BY qualification_type, study_level_group, intake_date_iso
ORDER BY intake_date_iso ASC, avg_nqf DESC, programme_count DESC;

-- Investigation 3: Window Function Ranking Highest-NQF Flagship Programmes per Faculty
WITH ranked_programmes AS (
    SELECT
        faculty_clean,
        programme_id,
        programme_title_clean,
        qualification_type,
        nqf_level,
        estimated_saqa_credits,
        delivery_mode_count,
        DENSE_RANK() OVER (
            PARTITION BY faculty_clean
            ORDER BY nqf_level DESC, estimated_saqa_credits DESC, programme_id ASC
        ) AS faculty_nqf_rank
    FROM eduvos_programmes
)
SELECT
    faculty_clean AS faculty,
    faculty_nqf_rank,
    programme_id,
    programme_title_clean,
    qualification_type,
    nqf_level,
    estimated_saqa_credits,
    delivery_mode_count
FROM ranked_programmes
WHERE faculty_nqf_rank <= 2
ORDER BY faculty_clean ASC, faculty_nqf_rank ASC, programme_id ASC;


﻿                                 version                                  
--------------------------------------------------------------------------
 PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit
(1 row)

 total_loaded_programmes 
-------------------------
                      68
(1 row)

       faculty       | total_programmes | foundational_cnt | undergrad_cnt | postgrad_cnt | corporate_cnt | avg_nqf_level | avg_saqa_credits | avg_delivery_modes | online_pct 
---------------------+------------------+------------------+---------------+--------------+---------------+---------------+------------------+--------------------+------------
 Technology          |               21 |                2 |            12 |            7 |             0 |          6.62 |            194.3 |               2.81 |      100.0
 Commerce            |               19 |                3 |             7 |            5 |             4 |          6.74 |            195.8 |           

### Investigation 1: Faculty Portfolio Summary

| faculty | total_programmes | foundational_cnt | undergrad_cnt | postgrad_cnt | corporate_cnt | avg_nqf_level | avg_saqa_credits | avg_delivery_modes | online_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Technology | 21 | 2 | 12 | 7 | 0 | 6.62 | 194.3 | 2.81 | 100 |
| Commerce | 19 | 3 | 7 | 5 | 4 | 6.74 | 195.8 | 2.21 | 100 |
| Humanities and Arts | 15 | 3 | 7 | 4 | 1 | 6.67 | 210 | 2.6 | 100 |
| Business School | 5 | 0 | 0 | 0 | 5 | 6 | 30 | 1 | 100 |
| Applied Sciences | 5 | 2 | 3 | 0 | 0 | 5.8 | 216 | 1.8 | 40 |
| Law | 3 | 0 | 2 | 1 | 0 | 8 | 360 | 2.67 | 100 |

### Investigation 2: Qualification Tier & Intake Cohort Structure

| qualification_type | study_level_group | intake_date_iso | avg_nqf | avg_years | programme_count | total_saqa_credits | pct_of_catalogue | nqf_imputed_cnt |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Corporate Training | Corporate & Short Courses | 2027-01-11 | 5.9 | 0.25 | 10 | 300 | 14.7 | 10 |
| Postgraduate Certificates & Diplomas | Postgraduate | 2027-01-15 | 8 | 1 | 1 | 120 | 1.5 | 0 |
| Access | Foundational | 2027-01-15 | 5 | 1 | 1 | 120 | 1.5 | 1 |
| Doctorates | Postgraduate | 2027-02-15 | 10 | 3 | 1 | 360 | 1.5 | 0 |
| Masters | Postgraduate | 2027-02-15 | 9 | 2 | 5 | 1200 | 7.4 | 0 |
| Honours | Postgraduate | 2027-02-15 | 8 | 1 | 8 | 960 | 11.8 | 0 |
| Postgraduate Certificates & Diplomas | Postgraduate | 2027-02-15 | 8 | 1 | 2 | 240 | 2.9 | 0 |
| Degree | Undergraduate | 2027-02-15 | 7 | 3.05 | 21 | 7680 | 30.9 | 0 |
| Higher Certificate | Undergraduate | 2027-02-15 | 5 | 1 | 10 | 1200 | 14.7 | 0 |
| Pre-Degree | Foundational | 2027-02-15 | 5 | 1 | 5 | 600 | 7.4 | 5 |
| Access | Foundational | 2027-02-15 | 5 | 1 | 4 | 480 | 5.9 | 4 |

![PostgreSQL ETL Analysis](../outputs/q4_etl_postgresql_analysis.png)
